In [1]:
#!/usr/bin/env python3
"""Interactive explorer of the UMAP/t-SNE layouts saved by npz_scanpy.py: joint or per-species views in 2D/3D, colouring by saved clusters, HDBSCAN, manual lasso clusters and exports.

Every .h5ad found one level below the npz_scanpy outdir of configs/param_configs.yaml is listed, and only the views (scope x dimension) whose layouts exist in it are offered, with one panel per available method.
"""
###################################### DEPENDENCIES_AND_CONSTANTS
import anndata as ad
import datetime
import hdbscan
from collections.abc import Callable
from IPython.display import clear_output, display
from ipywidgets import Button, Checkbox, ColorPicker, Dropdown, HBox, IntSlider, Layout, Output, Text, VBox
from matplotlib.lines import Line2D
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pyprojroot import find_root, has_file
import resource
import time
import traceback
import yaml
ALPHA = 0.75
ALPHA_INACTIVE = 0.025
COLOR_BY = ["(manual)", "species", "cluster", "intra_cluster"]
DEFAULT_COLOR = "#BFC5CE"
DEFAULT_OUTDIR = "data/dimred/"
DIMS = (2, 3)
DPI = 300
EXPORT_DIR = "viz/dimred"
FIGURE_HEIGHT = 620
FIGURE_WIDTH = 1100
LEGEND_FONT_SIZE = 9
LEGEND_NCOL = 4
LINE_COLOR = "black"
LINE_WIDTH = 0.8
MARKER_SIZE = 2
MARKER_SIZE_3D = 2
MAX_CLUSTERS = 15
METHODS = {"umap": "UMAP", "tsne": "t-SNE"}
MIN_CLUSTER_SIZE = 800
PALETTE = ["#ECDE90", "#93DEB6", "#F1B584", "#76BBE2", "#DA919B", "#899E64", "#8E81B9", "#A66C58", "#30767A", "#7A5271"]
SQUARE_SIZE = 8.0
TICK_FONT_FAMILY = "DejaVu Sans"
TICK_FONT_SIZE = 10
TITLE_FONT_SIZE = 14
YAML_PATH = "configs/param_configs.yaml"
ZOOM = {"umap": 0.9, "tsne": 1.10}
###################################### DISCOVERY
start_time = time.time()
PROJECT_ROOT = find_root(has_file(".here"), start=Path.cwd()).resolve()
def scanpy_outdir() -> Path:
    """Return the npz_scanpy outdir of the YAML config, or the default one.

    Returns:
        Path: Directory holding one sub-folder per run with its .h5ad.
    """
    cfg = PROJECT_ROOT / YAML_PATH
    ydata = (yaml.safe_load(cfg.read_text()) or {}) if cfg.exists() else {}
    return PROJECT_ROOT / ydata.get("npz_scanpy", {}).get("outdir", DEFAULT_OUTDIR)
def discover_views(outdir: Path) -> dict[str, tuple[Path, str, int, tuple[str, ...]]]:
    """List the available views, reading only obs and the obsm keys of each .h5ad.

    Args:
        outdir (Path): npz_scanpy output directory.

    Returns:
        dict[str, tuple[Path, str, int, tuple[str, ...]]]: Label -> (h5ad path, scope ``joint`` or species, dimension, methods).
    """
    views = {}
    for path in sorted(outdir.glob("*/*.h5ad")):
        adata = ad.read_h5ad(path, backed="r")
        keys, obs, run = set(adata.obsm.keys()), adata.obs, path.parent.name
        for d in DIMS:
            joint = tuple(m for m in METHODS if f"X_{m}_{d}d" in keys)
            if joint: views[f"{run} | joint | {d}D"] = (path, "joint", d, joint)
            intra = tuple(m for m in METHODS if f"X_{m}_{d}d_intra" in keys)
            for sp in obs["species"].unique() if intra else []:
                if obs.loc[obs["species"] == sp, "intra_cluster"].notna().any(): views[f"{run} | {sp} | {d}D"] = (path, sp, d, intra)
        adata.file.close()
    return views
OUTDIR = scanpy_outdir()
VIEWS = discover_views(OUTDIR)
if not VIEWS: raise FileNotFoundError(f"[PYTHON-ERROR] No .h5ad with UMAP/t-SNE layouts found in {OUTDIR}/<run>/; run slurms/supp_01_scanpy.sh first.")
print(f"[PYTHON-INFO] {len(VIEWS)} views found in {OUTDIR}")
CACHE = {}
###################################### WIDGETS
select_dropdown = Dropdown(options=[""] + list(VIEWS), description="View:", layout=Layout(width="420px"))
color_by_dropdown = Dropdown(options=COLOR_BY, value="(manual)", description="Color by:", layout=Layout(width="240px"))
reset_angles_btn = Button(description="Reset view", button_style="warning")
checklist_box = HBox([])
mcs_slider = IntSlider(value=MIN_CLUSTER_SIZE, min=100, max=10000, step=10, description="MinCS:", continuous_update=False, layout=Layout(width="320px"))
fit_on_dropdown = Dropdown(options=list(METHODS), value="umap", description="Fit on:")
fit_btn = Button(description="Fit HDBSCAN", button_style="primary")
cluster_name = Text(value="NewCluster", description="Name:", layout=Layout(width="210px"))
cluster_color = ColorPicker(value="#7b2c6d", description="Color:")
assign_btn = Button(description="Assign selected", button_style="primary")
rename_dropdown = Dropdown(options=[], description="Rename:", layout=Layout(width="210px"))
rename_text = Text(value="", description="To:", layout=Layout(width="160px"))
rename_btn = Button(description="Rename", button_style="info")
reset_btn = Button(description="Reset clusters", button_style="danger")
delete_btn = Button(description="Delete cluster", button_style="danger")
plot_fp_text = Text(value=EXPORT_DIR, description="Export dir:", layout=Layout(width="340px"))
export_tsv_btn = Button(description="Export TSV", button_style="info")
export_plot_btn = Button(description="Export Plot", button_style="info")
run_all_btn = Button(description="Run All", button_style="success")
output_area = Output()
plot_container = VBox([])
state = {"ids": None, "coords": {}, "methods": (), "obs": None, "cluster_labels": None, "cluster_info": {}, "filtered_mask": None, "current": None, "label_source": None, "checkboxes": {}, "fig_widget": None, "selected_indices": [], "last_selection_source": "umap", "display_title": "", "dim": 2}
###################################### DATA_AND_CLUSTERING
def load_view(label: str) -> tuple[pd.DataFrame, dict[str, np.ndarray]]:
    """Load the proteins and layouts of a view, caching each .h5ad once.

    Args:
        label (str): View label from ``VIEWS``.

    Returns:
        tuple[pd.DataFrame, dict[str, np.ndarray]]: Observations of the view and layout per method.
    """
    path, scope, d, methods = VIEWS[label]
    if path not in CACHE: CACHE[path] = ad.read_h5ad(path)
    adata = CACHE[path]
    mask = np.ones(adata.n_obs, dtype=bool) if scope == "joint" else (adata.obs["species"] == scope).values
    suffix = "" if scope == "joint" else "_intra"
    return adata.obs[mask].reset_index(drop=True), {m: np.asarray(adata.obsm[f"X_{m}_{d}d{suffix}"][mask], dtype=np.float64) for m in methods}
def view_tag(label: str) -> str:
    """Turn a view label into a file-name tag.

    Args:
        label (str): View label.

    Returns:
        str: Tag such as ``run_joint_2D``.
    """
    return "_".join(part.strip() for part in label.split("|"))
def fit_hdbscan_auto(coords: np.ndarray, min_cluster_size: int) -> tuple[np.ndarray, int]:
    """Fit HDBSCAN and relabel the MAX_CLUSTERS largest clusters 1..n by decreasing size, the rest 0.

    Args:
        coords (np.ndarray): Layout coordinates.
        min_cluster_size (int): HDBSCAN minimum cluster size.

    Returns:
        tuple[np.ndarray, int]: Labels and number of kept clusters.
    """
    mcs = int(min_cluster_size)
    if len(coords) < mcs: raise ValueError(f"Only {len(coords)} points available; MinCS is {mcs}")
    raw = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=min(10, mcs), metric="euclidean").fit_predict(coords)
    valid = raw[raw >= 0]
    if not len(valid): return np.zeros(len(coords), dtype=int), 0
    unique, counts = np.unique(valid, return_counts=True)
    mapping = {int(old): rank + 1 for rank, old in enumerate(unique[np.argsort(-counts)][:MAX_CLUSTERS])}
    return np.asarray([mapping.get(int(x), 0) for x in raw], dtype=int), len(mapping)
def palette_info(names: list[str]) -> dict[int, dict[str, str]]:
    """Build the cluster table with alternating PALETTE colours, id 0 being unassigned.

    Args:
        names (list[str]): Names of clusters 1..n.

    Returns:
        dict[int, dict[str, str]]: Cluster id -> name and colour.
    """
    return {0: {"name": "unassigned", "color": DEFAULT_COLOR}} | {i + 1: {"name": n, "color": PALETTE[i % len(PALETTE)]} for i, n in enumerate(names)}
def next_cluster_id() -> int:
    """Return the next free manual cluster id."""
    return max((int(cid) for cid in state["cluster_info"] if int(cid) != 0), default=0) + 1
def cluster_color_for(cid: int) -> str:
    """Return the colour of a cluster id."""
    return state["cluster_info"].get(int(cid), {"color": DEFAULT_COLOR})["color"]
def cluster_name_for(cid: int) -> str:
    """Return the name of a cluster id."""
    return state["cluster_info"].get(int(cid), {"name": "unassigned"})["name"]
###################################### PLOT_REFRESH
def update_checklist(labels: list[str]) -> None:
    """Rebuild the species visibility checkboxes.

    Args:
        labels (list[str]): Species of the current view.
    """
    state["checkboxes"] = {}
    for label in labels:
        cb = Checkbox(value=True, description=label, indent=False)
        cb.observe(lambda change: update_plot_traces(), names="value")
        state["checkboxes"][label] = cb
    checklist_box.children = list(state["checkboxes"].values())
def refresh_cluster_dropdown() -> None:
    """Refresh the rename/delete dropdown from the cluster table."""
    options = [f"{cid}: {info['name']}" for cid, info in sorted(state["cluster_info"].items()) if cid != 0]
    rename_dropdown.options = options
    rename_dropdown.value = options[0] if options else None
def update_plot_traces() -> None:
    """Recolour the data traces, fade hidden species and rebuild the legend traces."""
    fw, labels, sources = state["fig_widget"], state["cluster_labels"], state["label_source"]
    if fw is None or labels is None: return
    n_data = len(state["methods"])
    visible = {label for label, cb in state["checkboxes"].items() if cb.value}
    mask = np.isin(sources, list(visible)) if state["checkboxes"] else np.ones(len(labels), dtype=bool)
    state["filtered_mask"] = mask
    colors, alphas = [cluster_color_for(x) for x in labels], np.where(mask, ALPHA, ALPHA_INACTIVE)
    hover = [f"{src}<br>{cluster_name_for(cid)}<br>ID: {uid}" for src, cid, uid in zip(sources, labels, state["ids"])]
    scatter = go.Scatter3d if state["dim"] == 3 else go.Scatter
    empty = dict(x=[None], y=[None], z=[None]) if state["dim"] == 3 else dict(x=[None], y=[None])
    with fw.batch_update():
        for i in range(n_data):
            fw.data[i].text = hover
            if state["dim"] == 3:
                fw.data[i].marker.color = [f"rgba({int(c[1:3], 16)},{int(c[3:5], 16)},{int(c[5:7], 16)},{a})" for c, a in zip(colors, alphas)]
            else:
                fw.data[i].marker.color, fw.data[i].marker.opacity = colors, alphas
        fw.data = tuple(fw.data[:n_data])
        for sp in dict.fromkeys(sources.tolist()):
            fw.add_trace(scatter(**empty, mode="markers", marker=dict(size=8, color=DEFAULT_COLOR), name=f"{sp} (n={int(np.sum(sources == sp))})"))
        for cid in sorted((x for x in state["cluster_info"] if x != 0), key=lambda x: -int(np.sum(labels == x))):
            info = state["cluster_info"][cid]
            fw.add_trace(scatter(**empty, mode="markers", marker=dict(size=8, color=info["color"]), name=f"{info['name']} (n={int(np.sum(labels == cid))})"))
    refresh_cluster_dropdown()
###################################### SELECTION_AND_EDITING
def selection_callback(source: str) -> Callable:
    """Build a lasso/box/click callback storing the selected point indices.

    Args:
        source (str): Method of the trace (``umap`` or ``tsne``).

    Returns:
        Callable: Plotly trace callback.
    """
    def callback(trace: object, points: object, selector: object) -> None:
        """Store the selected indices of the trace."""
        indices = list(getattr(points, "point_inds", []) or [])
        if not indices: return
        state["selected_indices"] = sorted({int(i) for i in indices if 0 <= int(i) < len(state["ids"])})
        state["last_selection_source"] = source
        with output_area:
            clear_output(wait=True)
            print(f"[PYTHON-INFO] Selected {len(state['selected_indices'])} points from {METHODS[source]}.")
    return callback
def assign_selected(_: object = None) -> None:
    """Assign the selected points to a new manual cluster."""
    with output_area:
        clear_output(wait=True)
        indices = state["selected_indices"]
        if not indices or state["cluster_labels"] is None:
            print("[PYTHON-INFO] Select points in a plot first.")
            return
        cid = next_cluster_id()
        name = cluster_name.value.strip()
        if not name or name == "NewCluster": name = f"{'U' if state['last_selection_source'] == 'umap' else 'T'}-{cid:02d}"
        state["cluster_labels"][indices] = cid
        state["cluster_info"][cid] = {"name": name, "color": cluster_color.value}
        state["selected_indices"] = []
        update_plot_traces()
        print(f"[PYTHON-INFO] Assigned {len(indices)} points to cluster {cid} '{name}'.")
def rename_cluster(_: object = None) -> None:
    """Rename the cluster chosen in the dropdown."""
    with output_area:
        clear_output(wait=True)
        if not rename_dropdown.value or not rename_text.value.strip():
            print("[PYTHON-INFO] Choose a cluster and enter a new name.")
            return
        cid = int(rename_dropdown.value.split(":", 1)[0])
        state["cluster_info"][cid]["name"] = rename_text.value.strip()
        update_plot_traces()
        print(f"[PYTHON-INFO] Renamed cluster {cid} to '{rename_text.value.strip()}'.")
def reset_clusters(_: object = None) -> None:
    """Set every point back to unassigned."""
    with output_area:
        clear_output(wait=True)
        if state["ids"] is None:
            print("[PYTHON-INFO] Load a view first.")
            return
        state["cluster_labels"], state["cluster_info"], state["selected_indices"] = np.zeros(len(state["ids"]), dtype=int), palette_info([]), []
        update_plot_traces()
        print("[PYTHON-INFO] All points reset to unassigned.")
def delete_cluster(_: object = None) -> None:
    """Delete the cluster chosen in the dropdown, unassigning its points."""
    with output_area:
        clear_output(wait=True)
        if not rename_dropdown.value:
            print("[PYTHON-INFO] Choose a cluster first.")
            return
        cid = int(rename_dropdown.value.split(":", 1)[0])
        state["cluster_labels"][state["cluster_labels"] == cid] = 0
        del state["cluster_info"][cid]
        update_plot_traces()
        print(f"[PYTHON-INFO] Deleted cluster {cid}; its points are now unassigned.")
def reset_angles(_: object = None) -> None:
    """Reset the 3D cameras or the 2D axis ranges."""
    fw = state["fig_widget"]
    if fw is None: return
    if state["dim"] == 3:
        fw.update_layout({f"scene{'' if i == 1 else i}_camera": None for i in range(1, len(state["methods"]) + 1)})
    else:
        fw.update_xaxes(autorange=True)
        fw.update_yaxes(autorange=True)
###################################### LOADING_AND_PLOTTING
def zoom_range(values: np.ndarray, zoom: float) -> list[float]:
    """Return an axis range centred on the data and scaled by ``zoom``.

    Args:
        values (np.ndarray): Coordinates along one axis.
        zoom (float): Range multiplier.

    Returns:
        list[float]: Lower and upper bounds.
    """
    lo, hi = float(values.min()), float(values.max())
    mid, half = (lo + hi) / 2, (hi - lo) * zoom / 2
    return [mid - half, mid + half]
def build_figure() -> go.Figure:
    """Build one panel per available method of the current view, in 2D (lasso) or 3D (orbit)."""
    d, methods, title, n = state["dim"], state["methods"], state["display_title"], len(state["ids"])
    titles = [f"{title} - {METHODS[m]}" for m in methods]
    marker = dict(size=MARKER_SIZE if d == 2 else MARKER_SIZE_3D, color=[DEFAULT_COLOR] * n, opacity=ALPHA)
    if d == 2:
        fig = make_subplots(rows=1, cols=len(methods), subplot_titles=titles, horizontal_spacing=0.08)
        axis = dict(tickfont=dict(size=TICK_FONT_SIZE, family=TICK_FONT_FAMILY), ticks="outside", linecolor=LINE_COLOR, linewidth=LINE_WIDTH, showgrid=False, zeroline=False)
        for col, m in enumerate(methods, 1):
            c = state["coords"][m]
            fig.add_trace(go.Scattergl(x=c[:, 0], y=c[:, 1], mode="markers", marker=marker, customdata=state["ids"], name=f"{METHODS[m]} data", showlegend=False), row=1, col=col)
            fig.update_xaxes(title_text=f"{METHODS[m]}-1", range=zoom_range(c[:, 0], ZOOM[m]), **axis, row=1, col=col)
            fig.update_yaxes(title_text=f"{METHODS[m]}-2", range=zoom_range(c[:, 1], ZOOM[m]), **axis, row=1, col=col)
    else:
        fig = make_subplots(rows=1, cols=len(methods), specs=[[{"type": "scatter3d"}] * len(methods)], subplot_titles=titles, horizontal_spacing=0.04)
        axis = dict(backgroundcolor="white", showbackground=True, showline=True, linecolor=LINE_COLOR, linewidth=LINE_WIDTH, gridcolor="#DDDDDD", zerolinecolor="#BBBBBB", tickfont=dict(size=TICK_FONT_SIZE, family=TICK_FONT_FAMILY, color="black"))
        for col, m in enumerate(methods, 1):
            c = state["coords"][m]
            fig.add_trace(go.Scatter3d(x=c[:, 0], y=c[:, 1], z=c[:, 2], mode="markers", marker=marker, customdata=state["ids"], name=f"{METHODS[m]} data", showlegend=False), row=1, col=col)
            fig.update_layout({f"scene{'' if col == 1 else col}": {f"{ax}axis": dict(title=f"{METHODS[m]}-{i + 1}", **axis) for i, ax in enumerate("xyz")}})
    fig.update_layout(width=FIGURE_WIDTH, height=FIGURE_HEIGHT, dragmode="lasso" if d == 2 else "orbit", hovermode="closest", plot_bgcolor="white", paper_bgcolor="white", margin=dict(l=35, r=30, t=70, b=100), legend=dict(orientation="h", y=-0.16, x=0.5, xanchor="center", yanchor="top", bgcolor="rgba(0,0,0,0)"), font=dict(size=TICK_FONT_SIZE, family=TICK_FONT_FAMILY, color="#222222"))
    fig.update_annotations(font=dict(size=TITLE_FONT_SIZE, family=TICK_FONT_FAMILY, color="#111111"))
    return fig
def apply_color_by(_: object = None) -> None:
    """Colour the current view by a saved ``obs`` column, turning its categories into editable clusters."""
    column = color_by_dropdown.value
    if state["obs"] is None or column == "(manual)": return
    with output_area:
        clear_output(wait=True)
        values = state["obs"][column].astype("category").cat.remove_unused_categories()
        names = [str(c) for c in values.cat.categories]
        state["cluster_labels"], state["cluster_info"] = values.cat.codes.to_numpy().astype(int) + 1, palette_info([f"{column} {n}" if column != "species" else n for n in names])
        update_plot_traces()
        print(f"[PYTHON-INFO] Coloured by '{column}': {len(names)} groups.")
def load_and_plot(change: dict | None = None) -> None:
    """Load the selected view and draw it."""
    label = select_dropdown.value
    with output_area:
        clear_output(wait=True)
        if not label:
            print("[PYTHON-INFO] Select a view.")
            return
        try:
            obs, coords = load_view(label)
            ids, n = obs["protein_id"].astype(str).to_numpy(), len(obs)
            state.update(ids=ids, coords=coords, methods=tuple(coords), obs=obs, label_source=obs["species"].astype(str).to_numpy(), current=view_tag(label), dim=VIEWS[label][2], display_title=label.replace("|", "·"), selected_indices=[], cluster_labels=np.zeros(n, dtype=int), cluster_info=palette_info([]), filtered_mask=np.ones(n, dtype=bool))
            fit_on_dropdown.options = list(coords)
            update_checklist(list(dict.fromkeys(state["label_source"].tolist())))
            try: fw = go.FigureWidget(build_figure())
            except Exception as exc: raise RuntimeError("FigureWidget could not initialise: run `pip install -U plotly anywidget ipywidgets` and restart the kernel.") from exc
            for i, m in enumerate(coords):
                fw.data[i].on_selection(selection_callback(m))
                fw.data[i].on_click(selection_callback(m))
            state["fig_widget"] = fw
            plot_container.children = [fw]
            update_plot_traces()
            apply_color_by()
            print(f"[PYTHON-INFO] Loaded {n:,} points for '{label}'. Lasso/box select in 2D; clicks also select points.")
        except Exception as exc:
            print(f"[PYTHON-ERROR] {exc}")
            traceback.print_exc()
def fit_clusters(_: object = None) -> None:
    """Fit HDBSCAN on the chosen layout of the current view."""
    with output_area:
        clear_output(wait=True)
        if state["ids"] is None:
            print("[PYTHON-INFO] Load a view first.")
            return
        method = fit_on_dropdown.value
        labels, n_clusters = fit_hdbscan_auto(state["coords"][method], mcs_slider.value)
        prefix = "U" if method == "umap" else "T"
        state["cluster_labels"], state["cluster_info"] = labels, palette_info([f"{prefix}-{cid:02d}" for cid in range(1, n_clusters + 1)])
        update_plot_traces()
        print(f"[PYTHON-INFO] {METHODS[method]}: {n_clusters} cluster(s) found.")
###################################### EXPORTS
def output_directory() -> Path:
    """Create and return the export directory, relative paths being taken from the project root."""
    raw = plot_fp_text.value.strip()
    if not raw: raise ValueError("Specify an export directory.")
    path = Path(raw) if Path(raw).is_absolute() else PROJECT_ROOT / raw
    path.mkdir(parents=True, exist_ok=True)
    return path
def export_results(_: object = None) -> None:
    """Export the cluster labels of the current view (selection flagged) to TSV."""
    with output_area:
        clear_output(wait=True)
        if state["ids"] is None:
            print("[PYTHON-INFO] Load a view first.")
            return
        selected = np.zeros(len(state["ids"]), dtype=bool)
        if state["selected_indices"]: selected[state["selected_indices"]] = True
        else: selected = state["filtered_mask"]
        df = pd.DataFrame({"id": state["ids"], "species": state["label_source"], "cluster_id": state["cluster_labels"], "cluster_name": [cluster_name_for(x) for x in state["cluster_labels"]], "view": state["current"], "in_selection": selected})
        path = output_directory() / f"{state['current']}_clusters.tsv"
        df.to_csv(path, sep="\t", index=False)
        print(f"[PYTHON-INFO] Exported {len(df):,} rows to {path}")
def export_plot(_: object = None) -> None:
    """Export the current view (matplotlib in 2D, Kaleido in 3D) and its horizontal and vertical legends."""
    with output_area:
        clear_output(wait=True)
        if state["fig_widget"] is None:
            print("[PYTHON-INFO] Load a view first.")
            return
        out_dir, tag, labels, methods = output_directory(), state["current"], state["cluster_labels"], state["methods"]
        colors = [cluster_color_for(x) for x in labels]
        if state["dim"] == 2:
            fig, axes = plt.subplots(1, len(methods), figsize=(SQUARE_SIZE * len(methods) / 2, SQUARE_SIZE / 2), dpi=DPI, squeeze=False)
            for ax, m in zip(axes[0], methods):
                c = state["coords"][m]
                ax.scatter(c[:, 0], c[:, 1], c=colors, s=MARKER_SIZE, alpha=ALPHA, linewidths=0, rasterized=True)
                ax.set_xlim(*zoom_range(c[:, 0], ZOOM[m]))
                ax.set_ylim(*zoom_range(c[:, 1], ZOOM[m]))
                ax.set_title(f"{state['display_title']} - {METHODS[m]}", fontsize=TITLE_FONT_SIZE, fontname=TICK_FONT_FAMILY)
                ax.set_xlabel(f"{METHODS[m]}-1", fontsize=TICK_FONT_SIZE, fontname=TICK_FONT_FAMILY)
                ax.set_ylabel(f"{METHODS[m]}-2", fontsize=TICK_FONT_SIZE, fontname=TICK_FONT_FAMILY)
                ax.tick_params(axis="both", labelsize=TICK_FONT_SIZE, length=4, width=LINE_WIDTH, colors="black")
                for spine in ax.spines.values(): spine.set(color=LINE_COLOR, linewidth=LINE_WIDTH)
            fig.tight_layout()
            for ext in ("pdf", "png"): fig.savefig(out_dir / f"{tag}_plot.{ext}", dpi=DPI, bbox_inches="tight", facecolor="white")
            plt.close(fig)
        else:
            fig = go.Figure(state["fig_widget"].to_dict())
            fig.update_layout(width=1500, height=900)
            try:
                for ext in ("pdf", "png"): fig.write_image(str(out_dir / f"{tag}_plot.{ext}"), scale=2 if ext == "png" else 1)
            except Exception as exc:
                print(f"[PYTHON-ERROR] Static 3D export needs Kaleido (`pip install -U kaleido`): {exc}")
                return
        handles = [Line2D([0], [0], marker="o", linestyle="", color=info["color"], label=f"{info['name']} (n={int(np.sum(labels == cid))})", markersize=5) for cid, info in sorted(state["cluster_info"].items()) if cid != 0]
        handles.append(Line2D([0], [0], marker="o", linestyle="", color=DEFAULT_COLOR, label=f"Unassigned (n={int(np.sum(labels == 0))})", markersize=5))
        for orientation in ("horizontal", "vertical"):
            legend_fig = plt.figure(figsize=(10, 1.8) if orientation == "horizontal" else (4, max(2, len(handles) * 0.35)), dpi=DPI)
            legend_fig.legend(handles=handles, loc="center", ncol=LEGEND_NCOL if orientation == "horizontal" else 1, frameon=False, fontsize=LEGEND_FONT_SIZE)
            for ext in ("pdf", "png"): legend_fig.savefig(out_dir / f"{tag}_legend_{orientation}.{ext}", dpi=DPI, bbox_inches="tight", facecolor="white")
            plt.close(legend_fig)
        print(f"[PYTHON-INFO] Saved plot and legends to {out_dir}")
###################################### BATCH
def run_all(_: object = None) -> None:
    """Fit HDBSCAN on every view whose layouts include the chosen method and export one TSV per view."""
    with output_area:
        clear_output(wait=True)
        out_dir, method, done = output_directory(), fit_on_dropdown.value, 0
        todo = [label for label, (*_, methods) in VIEWS.items() if method in methods]
        for label in todo:
            try:
                obs, coords = load_view(label)
                labels, n_clusters = fit_hdbscan_auto(coords[method], mcs_slider.value)
                names = palette_info([f"{'U' if method == 'umap' else 'T'}-{cid:02d}" for cid in range(1, n_clusters + 1)])
                pd.DataFrame({"id": obs["protein_id"], "species": obs["species"], "cluster_id": labels, "cluster_name": [names[int(x)]["name"] for x in labels], "view": view_tag(label)}).to_csv(out_dir / f"{view_tag(label)}_clusters.tsv", sep="\t", index=False)
                done += 1
                print(f"[PYTHON-INFO] [{done}/{len(todo)}] {label}: {n_clusters} clusters")
            except Exception as exc:
                print(f"[PYTHON-ERROR] {label}: {exc}")
        print(f"[PYTHON-INFO] Batch complete: {done}/{len(todo)} succeeded.")
###################################### BINDINGS_AND_DISPLAY
select_dropdown.observe(load_and_plot, names="value")
color_by_dropdown.observe(apply_color_by, names="value")
reset_angles_btn.on_click(reset_angles)
fit_btn.on_click(fit_clusters)
assign_btn.on_click(assign_selected)
rename_btn.on_click(rename_cluster)
reset_btn.on_click(reset_clusters)
delete_btn.on_click(delete_cluster)
export_tsv_btn.on_click(export_results)
export_plot_btn.on_click(export_plot)
run_all_btn.on_click(run_all)
display(VBox([HBox([select_dropdown, color_by_dropdown, reset_angles_btn]), checklist_box, HBox([mcs_slider, fit_on_dropdown, fit_btn]), HBox([cluster_name, cluster_color, assign_btn]), HBox([rename_dropdown, rename_text, rename_btn, delete_btn, reset_btn]), HBox([plot_fp_text, export_tsv_btn, export_plot_btn, run_all_btn]), plot_container, output_area]))
print(f"[PYTHON-INFO] {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} {time.time() - start_time:.2f}s {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024:.2f}MB")

FileNotFoundError: [PYTHON-ERROR] No .h5ad with UMAP/t-SNE layouts found in /orfeo/cephfs/scratch/area/ssenci/senci_projects/REPR_LRN/SCambrian/data/scanpy/<run>/; run slurms/supp_01_scanpy.sh first.